In [1]:
import duckdb

con = duckdb.connect("clickstream.duckdb")


In [ ]:
#Row Count
con.sql("SELECT COUNT(*) AS row_count FROM clickstream").show()

In [ ]:
#Type distribution
con.sql("""
    SELECT type, COUNT(*) AS row_count, SUM(n) AS total_n
    FROM clickstream
    GROUP BY type
    ORDER BY total_n DESC
""").show()

In [ ]:
# Min/max/avg n (overall)
con.sql("SELECT MIN(n) AS min_n, MAX(n) AS max_n, AVG(n) AS avg_n FROM clickstream").show()

In [ ]:
con.sql("""
    SELECT COUNT(*) AS redlink_rows
    FROM clickstream
    WHERE type = 'redlink'
""").show()

In [ ]:

#    other-internal vs. other-wikipedia, etc.)
con.sql("""
    SELECT prev, COUNT(*) AS row_count, SUM(n) AS total_n
    FROM clickstream
    WHERE prev LIKE 'other%'
    GROUP BY prev
    ORDER BY total_n DESC
""").show()

In [ ]:
# 6. Suppression floor check — confirm the min n is 10, 11, 12, etc.
con.sql("""
    SELECT n, COUNT(*) AS row_count
    FROM clickstream
    GROUP BY n
    ORDER BY n ASC
    LIMIT 10
""").show()

In [2]:
con.sql("""SELECT
    curr,
    external_n,
    internal_n,
    external_n + internal_n AS total_n,
    ROUND(external_n * 1.0 / NULLIF(external_n + internal_n, 0), 4) AS external_share
FROM (
    SELECT
        curr,
        SUM(CASE WHEN prev LIKE 'other%' AND prev != 'other-internal' THEN n ELSE 0 END) AS external_n,
        SUM(CASE WHEN type IN ('link', 'other') OR prev = 'other-internal' THEN n ELSE 0 END) AS internal_n
    FROM clickstream
    GROUP BY curr
    HAVING SUM(CASE WHEN prev LIKE 'other%' AND prev != 'other-internal' THEN n ELSE 0 END)
         + SUM(CASE WHEN type IN ('link', 'other') OR prev = 'other-internal' THEN n ELSE 0 END) >= 2113
)
ORDER BY total_n DESC""").show()

┌─────────────────────────────────────┬────────────┬────────────┬───────────┬────────────────┐
│                curr                 │ external_n │ internal_n │  total_n  │ external_share │
│               varchar               │   int128   │   int128   │  int128   │     double     │
├─────────────────────────────────────┼────────────┼────────────┼───────────┼────────────────┤
│ Main_Page                           │  200876997 │   15338678 │ 216215675 │         0.9291 │
│ 2026_FIFA_World_Cup                 │   14593610 │    4347615 │  18941225 │         0.7705 │
│ Hyphen-minus                        │    8702148 │     166316 │   8868464 │         0.9812 │
│ Obsession_(2025_film)               │    5009214 │     711380 │   5720594 │         0.8756 │
│ Oliver_Tree                         │    4453453 │     382053 │   4835506 │          0.921 │
│ FIFA_World_Cup                      │    3975495 │     852330 │   4827825 │         0.8235 │
│ .xxx                                │    4809376

In [6]:
#important query, asks the actual interesting question about this dataset
con.sql("""SELECT
    prev,
    curr,
    n
FROM clickstream
WHERE type = 'other'
  AND prev != 'Main_Page'
  AND curr != 'Main_Page'
ORDER BY n DESC
LIMIT 50""").show(max_rows=50)

┌─────────────────────────────────────────┬──────────────────────────────────────┬───────┐
│                  prev                   │                 curr                 │   n   │
│                 varchar                 │               varchar                │ int32 │
├─────────────────────────────────────────┼──────────────────────────────────────┼───────┤
│ UBlock_Origin                           │ Google_Search                        │ 52755 │
│ Marjane_Satrapi                         │ Broken_heart                         │ 40002 │
│ Zion_Suzuki                             │ Ghanaians                            │ 39778 │
│ 2026_in_film                            │ Dear_You_(film)                      │ 34791 │
│ Peddi                                   │ List_of_Telugu_films_of_2026         │ 27410 │
│ Amelia_Dimoldenberg                     │ Zara_Larsson                         │ 26891 │
│ 2026_French_Open                        │ 2025_French_Open_–_Men's_singles     │ 23018 │